# 신호등 없는 횡단보도 × 노인 보행 사고 — 서울 25구

**목표**: 서울 25구 횡단보도(40,281건) 중 **신호등이 없는 횡단보도**와 TAAS 노인 보행사고(2024)를
공간 결합해, 신호 없는 횡단보도가 노인 사고와 어떻게 연관되는지 자치구 단위로 본다.

**입력**
- `서울특별시_자치구_횡단보도_20260320.csv` — 서울시 25구 전체 횡단보도 (40,281건)
  - 좌표(경도/위도) · 보행등 유무 · 음향신호기 · 고원식 · 보행자작동신호기
  - **녹색신호 시간/연장 데이터 없음** → 신호 *유무* 중심 분석
- `data/processed/taas_seoul_elderly_pedestrian_2024.csv` — TAAS 65세+ 보행 사고 (1,963건)

**산출**
- 자치구별 통계 테이블 (횡단보도 / 신호 보급률 / 노인 사고) — 노트북 내 print
- 인프라 결핍도(신호+음향+고원식 z-score 합) 자치구 랭킹 — 노트북 내 print
- `crosswalk_safety_map.html` — 카카오맵 (사고 + 신호 없는 횡단보도)


In [8]:
# ============================================================
# 셀 1 - 환경
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

CROSSWALK_CSV = '서울특별시_자치구_횡단보도_20260320.csv'
TAAS_CSV = 'data/processed/taas_seoul_elderly_pedestrian_2024.csv'

print(f"횡단보도 CSV: {CROSSWALK_CSV}")
print(f"TAAS CSV   : {TAAS_CSV}")


횡단보도 CSV: 서울특별시_자치구_횡단보도_20260320.csv
TAAS CSV   : data/processed/taas_seoul_elderly_pedestrian_2024.csv


In [9]:
# ============================================================
# 셀 2 - 횡단보도 CSV 로드 (서울 25구만)
# ============================================================
cw = pd.read_csv(CROSSWALK_CSV)
print(f"전체 행: {len(cw):,}")
print(f"컬럼: {list(cw.columns)}")

cw = cw[cw['시도명'] == '서울특별시'].copy()

# ⚠️ 주의: 이 CSV는 컬럼 헤더가 뒤바뀌어 있음
#   '경도' 컬럼 값 = 37.x (실제로는 위도)
#   '위도' 컬럼 값 = 127.x (실제로는 경도)
# → 따라서 '경도' → lat, '위도' → lon 으로 매핑
cw = cw.rename(columns={'경도': 'lat', '위도': 'lon', '시군구명': '구'})
cw = cw.dropna(subset=['lat', 'lon', '구']).reset_index(drop=True)

# 좌표 검증 (서울 lat ~37.4-37.7, lon ~126.7-127.2)
assert 37.0 < cw['lat'].mean() < 38.0, f"lat 이상함: {cw['lat'].mean()}"
assert 126.0 < cw['lon'].mean() < 128.0, f"lon 이상함: {cw['lon'].mean()}"

for col in ['보행등유무', '음향신호기설치여부', '보행자작동신호기유무', '고원식횡단보도유무']:
    cw[col] = cw[col].fillna('N').str.upper().eq('Y')

cw['신호없음'] = ~cw['보행등유무']

print(f"\n서울 횡단보도: {len(cw):,}건")
print(f"  - lat 범위: {cw['lat'].min():.4f} ~ {cw['lat'].max():.4f}")
print(f"  - lon 범위: {cw['lon'].min():.4f} ~ {cw['lon'].max():.4f}")
print(f"  - 보행등 있음: {cw['보행등유무'].sum():,} ({cw['보행등유무'].mean()*100:.1f}%)")
print(f"  - 보행등 없음: {cw['신호없음'].sum():,} ({cw['신호없음'].mean()*100:.1f}%)")
print(f"  - 음향신호기 : {cw['음향신호기설치여부'].sum():,} ({cw['음향신호기설치여부'].mean()*100:.1f}%)")
print(f"  - 고원식     : {cw['고원식횡단보도유무'].sum():,} ({cw['고원식횡단보도유무'].mean()*100:.1f}%)")
print(f"  - 작동식 신호: {cw['보행자작동신호기유무'].sum():,} ({cw['보행자작동신호기유무'].mean()*100:.1f}%)")
print(f"\n자치구 수: {cw['구'].nunique()}")


전체 행: 40,281
컬럼: ['연번', '시도명', '시군구명', '소재지지번주소', '횡단보도관리번호', '횡단보도종류', '고원식횡단보도유무', '경도', '위도', '보행등유무', '음향신호기설치여부', '보행자작동신호기유무', '데이터기준일자']

서울 횡단보도: 40,280건
  - lat 범위: 37.4339 ~ 37.6922
  - lon 범위: 126.7688 ~ 127.1828
  - 보행등 있음: 13,545 (33.6%)
  - 보행등 없음: 26,735 (66.4%)
  - 음향신호기 : 11,550 (28.7%)
  - 고원식     : 2,075 (5.2%)
  - 작동식 신호: 313 (0.8%)

자치구 수: 25


In [10]:
# ============================================================
# 셀 3 - TAAS 노인 보행사고 로드
# ============================================================
acc = pd.read_csv(TAAS_CSV)
acc['구'] = acc['legaldong_name'].str.extract(r'서울특별시 (\S+구)')[0]
acc = acc.dropna(subset=['lat', 'lon', '구']).reset_index(drop=True)

acc['횡단중'] = acc['acdnt_dc'].fillna('').str.contains('횡단')

print(f"TAAS 노인 보행사고: {len(acc):,}건")
print(f"  - 횡단 관련       : {acc['횡단중'].sum():,} ({acc['횡단중'].mean()*100:.1f}%)")
print(f"  - 자치구          : {acc['구'].nunique()}개")
print(f"\n사고 유형 분포 (acdnt_dc):")
print(acc['acdnt_dc'].value_counts().head(8).to_string())


TAAS 노인 보행사고: 1,963건
  - 횡단 관련       : 620 (31.6%)
  - 자치구          : 25개

사고 유형 분포 (acdnt_dc):
acdnt_dc
차대사람 - 기타            878
차대사람 - 횡단중           620
차대사람 - 차도통행중         216
차대사람 - 보도통행중         139
차대사람 - 길가장자리구역통행중     88
차대차 - 기타              14
차대차 - 충돌               7
차대차 - 추돌               1


In [11]:
# ============================================================
# 셀 4 - BallTree 공간 결합 (사고 ↔ 가장 가까운 신호 없는 횡단보도)
# ============================================================
EARTH_R = 6371000  # m

def nearest_dist_m(query_pts, ref_pts):
    if len(ref_pts) == 0 or len(query_pts) == 0:
        return np.full(len(query_pts), np.inf), np.full(len(query_pts), -1)
    tree = BallTree(np.radians(ref_pts), metric='haversine')
    d, idx = tree.query(np.radians(query_pts), k=1)
    return (d.flatten() * EARTH_R), idx.flatten()

acc_pts = acc[['lat', 'lon']].values
cw_all_pts = cw[['lat', 'lon']].values
cw_no_pts  = cw.loc[cw['신호없음'], ['lat', 'lon']].values

acc['최근접_횡단보도_m'], _ = nearest_dist_m(acc_pts, cw_all_pts)
acc['최근접_신호없음_m'], _ = nearest_dist_m(acc_pts, cw_no_pts)

acc['인접_신호없음_50m']  = acc['최근접_신호없음_m'] <= 50
acc['인접_신호없음_100m'] = acc['최근접_신호없음_m'] <= 100
acc['인접_횡단보도_50m']  = acc['최근접_횡단보도_m'] <= 50

print("=== 사고 지점 ↔ 가장 가까운 횡단보도 거리 (m) ===")
print(acc[['최근접_횡단보도_m', '최근접_신호없음_m']].describe(percentiles=[.25,.5,.75,.9]).round(1))

print(f"\n노인 사고 인접 횡단보도 비율:")
print(f"  - 50m  내 어떤 횡단보도 있음: {acc['인접_횡단보도_50m'].sum():,} ({acc['인접_횡단보도_50m'].mean()*100:.1f}%)")
print(f"  - 50m  내 신호 없는 횡단보도: {acc['인접_신호없음_50m'].sum():,} ({acc['인접_신호없음_50m'].mean()*100:.1f}%)")
print(f"  - 100m 내 신호 없는 횡단보도: {acc['인접_신호없음_100m'].sum():,} ({acc['인접_신호없음_100m'].mean()*100:.1f}%)")

횡단_acc = acc[acc['횡단중']]
print(f"\n횡단 관련 사고 ({len(횡단_acc):,}건) 만:")
print(f"  - 50m  내 신호 없는 횡단보도: {횡단_acc['인접_신호없음_50m'].sum():,} ({횡단_acc['인접_신호없음_50m'].mean()*100:.1f}%)")


=== 사고 지점 ↔ 가장 가까운 횡단보도 거리 (m) ===
       최근접_횡단보도_m  최근접_신호없음_m
count      1963.0      1963.0
mean         41.0        54.5
std          52.8        58.4
min           0.2         0.2
25%           7.9        12.5
50%          19.6        35.5
75%          55.9        77.3
90%         110.0       129.4
max         672.1       672.1

노인 사고 인접 횡단보도 비율:
  - 50m  내 어떤 횡단보도 있음: 1,418 (72.2%)
  - 50m  내 신호 없는 횡단보도: 1,180 (60.1%)
  - 100m 내 신호 없는 횡단보도: 1,630 (83.0%)

횡단 관련 사고 (620건) 만:
  - 50m  내 신호 없는 횡단보도: 473 (76.3%)


In [12]:
# ============================================================
# 셀 5 - 자치구별 통합 통계 테이블
# ============================================================
cw_gu = cw.groupby('구').agg(
    횡단보도_총=('보행등유무', 'count'),
    신호있음=('보행등유무', 'sum'),
    신호없음=('신호없음', 'sum'),
    음향신호기=('음향신호기설치여부', 'sum'),
    고원식=('고원식횡단보도유무', 'sum'),
    작동식=('보행자작동신호기유무', 'sum'),
).reset_index()
cw_gu['신호보급률'] = cw_gu['신호있음'] / cw_gu['횡단보도_총']
cw_gu['음향보급률'] = cw_gu['음향신호기'] / cw_gu['횡단보도_총']
cw_gu['고원식보급률'] = cw_gu['고원식'] / cw_gu['횡단보도_총']

acc_gu = acc.groupby('구').agg(
    노인사고=('acdnt_no', 'count'),
    횡단사고=('횡단중', 'sum'),
    인접신호없음_50m=('인접_신호없음_50m', 'sum'),
).reset_index()
acc_gu['신호없음근접비율'] = acc_gu['인접신호없음_50m'] / acc_gu['노인사고']

stats = cw_gu.merge(acc_gu, on='구', how='outer').fillna(0)
stats = stats.sort_values('노인사고', ascending=False).reset_index(drop=True)

print("=== 자치구별 횡단보도 인프라 + 노인 사고 통합 ===\n")
display_cols = ['구', '횡단보도_총', '신호없음', '신호보급률', '음향보급률',
                '노인사고', '횡단사고', '인접신호없음_50m', '신호없음근접비율']
print(stats[display_cols].round(3).to_string(index=False))

print(f"\n--- 요약 ---")
print(f"신호 보급률 평균: {stats['신호보급률'].mean()*100:.1f}% (최저 {stats['신호보급률'].min()*100:.1f}% / 최고 {stats['신호보급률'].max()*100:.1f}%)")
print(f"음향 보급률 평균: {stats['음향보급률'].mean()*100:.1f}%")
print(f"\n신호 없는 횡단보도 50m 내 노인 사고 발생 자치구 TOP5:")
print(stats.nlargest(5, '인접신호없음_50m')[['구', '인접신호없음_50m', '노인사고', '신호없음근접비율']].to_string(index=False))


=== 자치구별 횡단보도 인프라 + 노인 사고 통합 ===

   구  횡단보도_총  신호없음  신호보급률  음향보급률  노인사고  횡단사고  인접신호없음_50m  신호없음근접비율
동대문구    1479   988  0.332  0.283   130    37          61     0.469
 송파구    2324  1377  0.407  0.338   116    33          67     0.578
 은평구    1663  1078  0.352  0.316   111    41          66     0.595
 강남구    2745  1955  0.288  0.248   110    29          72     0.655
 성북구    1859  1314  0.293  0.239   103    34          69     0.670
 강동구    1808  1064  0.412  0.338   100    21          56     0.560
 구로구    1810  1222  0.325  0.275    97    30          56     0.577
 강북구    1313   943  0.282  0.235    94    33          60     0.638
 강서구    2179  1306  0.401  0.363    92    29          48     0.522
 중랑구    1659  1131  0.318  0.278    88    25          45     0.511
 양천구    1748  1148  0.343  0.320    87    27          59     0.678
 노원구    1706  1026  0.399  0.319    85    30          51     0.600
 관악구    1039   631  0.393  0.329    82    15          49     0.598
 동작구    1025   710  0.307  0

In [13]:
# ============================================================
# 셀 6 - 인프라 결핍도 점수 (자치구별 z-score 합)
# ============================================================
def z(s):
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

stats['결핍_신호'] = z(1 - stats['신호보급률'])
stats['결핍_음향'] = z(1 - stats['음향보급률'])
stats['결핍_고원식'] = z(1 - stats['고원식보급률'])
stats['인프라_결핍도'] = stats['결핍_신호'] + stats['결핍_음향'] + stats['결핍_고원식']

print("=== 인프라 결핍도 TOP 5 (인프라 부족 자치구) ===")
print(stats.nlargest(5, '인프라_결핍도')[
    ['구', '신호보급률', '음향보급률', '고원식보급률', '인프라_결핍도', '노인사고']
].round(3).to_string(index=False))

print("\n=== 인프라 결핍도 BOTTOM 5 (인프라 양호 자치구) ===")
print(stats.nsmallest(5, '인프라_결핍도')[
    ['구', '신호보급률', '음향보급률', '고원식보급률', '인프라_결핍도', '노인사고']
].round(3).to_string(index=False))

print("\n=== 결핍도 vs 사고 — 자치구 전체 ===")
print(stats[['구', '인프라_결핍도', '노인사고', '횡단사고', '신호없음근접비율']]
      .sort_values('인프라_결핍도', ascending=False).round(3).to_string(index=False))


=== 인프라 결핍도 TOP 5 (인프라 부족 자치구) ===
  구  신호보급률  음향보급률  고원식보급률  인프라_결핍도  노인사고
 중구  0.218  0.205   0.029    5.529    61
강남구  0.288  0.248   0.043    2.383   110
종로구  0.303  0.258   0.030    2.171    60
강북구  0.282  0.235   0.067    2.119    94
중랑구  0.318  0.278   0.009    1.949    88

=== 인프라 결핍도 BOTTOM 5 (인프라 양호 자치구) ===
  구  신호보급률  음향보급률  고원식보급률  인프라_결핍도  노인사고
송파구  0.407  0.338   0.096   -4.358   116
강동구  0.412  0.338   0.071   -3.687   100
강서구  0.401  0.363   0.007   -2.133    92
노원구  0.399  0.319   0.032   -1.675    85
은평구  0.352  0.316   0.065   -1.567   111

=== 결핍도 vs 사고 — 자치구 전체 ===
   구  인프라_결핍도  노인사고  횡단사고  신호없음근접비율
  중구    5.529    61    23     0.738
 강남구    2.383   110    29     0.655
 종로구    2.171    60    25     0.667
 강북구    2.119    94    33     0.638
 중랑구    1.949    88    25     0.511
 서초구    1.745    47    15     0.702
 동작구    1.594    82    29     0.573
 구로구    1.140    97    30     0.577
영등포구    1.052    79    37     0.671
 마포구    0.762    38    12     0.632
 성북구    0.

In [14]:
# ============================================================
# 셀 7 - 카카오맵 HTML (사고 + 신호 없는 횡단보도)
# ============================================================
import json as _json

kakao_key = "4827d1df867dfc08ae1daba2b1d25835"

acc_pts_js = acc[['lat','lon','구','acdnt_dc','acdnt_gae_dc','횡단중']].copy()
acc_pts_js.columns = ['lat','lon','gu','acdnt','grade','crosswalk']

# 신호 없는 횡단보도 — 사고 200m 내만 (브라우저 부담 줄이기)
cw_no = cw[cw['신호없음']].copy()
cw_no_pts = cw_no[['lat','lon']].values
acc_pts_arr = acc[['lat','lon']].values
if len(cw_no_pts) > 0 and len(acc_pts_arr) > 0:
    tree = BallTree(np.radians(acc_pts_arr), metric='haversine')
    d, _ = tree.query(np.radians(cw_no_pts), k=1)
    cw_no['최근접_사고_m'] = d.flatten() * 6371000
    cw_no_near = cw_no[cw_no['최근접_사고_m'] <= 200].copy()
else:
    cw_no_near = cw_no.head(0).copy()

cw_js = cw_no_near[['lat','lon','구','음향신호기설치여부','고원식횡단보도유무']].copy()
cw_js.columns = ['lat','lon','gu','음향','고원식']
cw_js['음향'] = cw_js['음향'].astype(int)
cw_js['고원식'] = cw_js['고원식'].astype(int)

print(f"지도 표시:")
print(f"  - 노인 사고: {len(acc_pts_js):,}")
print(f"  - 신호 없는 횡단보도(사고 200m 내): {len(cw_js):,} / 전체 신호없음 {len(cw_no):,}")

acc_json = acc_pts_js.to_dict('records')
cw_json = cw_js.to_dict('records')
gu_list = sorted(set(acc_pts_js['gu']) | set(cw_js['gu']))

html = """<!DOCTYPE html>
<html lang="ko"><head>
<meta charset="utf-8">
<title>서울 신호 없는 횡단보도 × 노인 보행사고</title>
<script src="//dapi.kakao.com/v2/maps/sdk.js?appkey=__KAKAO__"></script>
<style>
body{margin:0;font-family:-apple-system,BlinkMacSystemFont,sans-serif}
#map{width:100%;height:100vh}
#panel{position:absolute;top:12px;left:12px;background:rgba(255,255,255,0.96);
       padding:14px 16px;border-radius:8px;box-shadow:0 2px 12px rgba(0,0,0,0.18);z-index:10;max-width:300px}
h3{margin:0 0 8px 0;font-size:15px}
.legend{display:flex;align-items:center;gap:6px;margin:4px 0;font-size:13px}
.dot{width:12px;height:12px;border-radius:50%;display:inline-block}
.red{background:#d62728}.blue{background:#1f77b4}
select{width:100%;margin-top:8px;padding:5px;font-size:13px}
.stat{font-size:12px;color:#444;margin-top:8px;line-height:1.6}
</style>
</head>
<body>
<div id="panel">
  <h3>서울 신호 없는 횡단보도 × 노인 사고</h3>
  <div class="legend"><span class="dot red"></span>노인 보행사고 (TAAS 2024)</div>
  <div class="legend"><span class="dot blue"></span>신호 없는 횡단보도 (사고 200m 내)</div>
  <select id="guSelect"><option value="">전체 자치구</option>__OPTIONS__</select>
  <div class="stat" id="stat"></div>
</div>
<div id="map"></div>
<script>
const ACC = __ACC__;
const CW  = __CW__;

const map = new kakao.maps.Map(document.getElementById("map"), {
  center: new kakao.maps.LatLng(37.55, 126.99),
  level: 7
});

const accLayer = [];
const cwLayer = [];

function clear(arr){arr.forEach(m=>m.setMap(null));arr.length=0}

function draw(filter){
  clear(accLayer); clear(cwLayer);
  let accCnt=0, cwCnt=0;
  ACC.forEach(p=>{
    if(filter && p.gu!==filter) return;
    const c = new kakao.maps.Circle({
      center: new kakao.maps.LatLng(p.lat, p.lon),
      radius: p.crosswalk ? 12 : 8,
      strokeWeight: 0, fillColor:"#d62728",
      fillOpacity: p.crosswalk ? 0.7 : 0.4,
    });
    c.setMap(map); accLayer.push(c); accCnt++;
  });
  CW.forEach(p=>{
    if(filter && p.gu!==filter) return;
    const c = new kakao.maps.Circle({
      center: new kakao.maps.LatLng(p.lat, p.lon),
      radius: 7, strokeWeight: 0, fillColor:"#1f77b4", fillOpacity:0.5,
    });
    c.setMap(map); cwLayer.push(c); cwCnt++;
  });
  document.getElementById("stat").textContent =
    `노인 사고 ${accCnt}건 · 신호 없는 횡단보도 ${cwCnt}개 표시 중`;
}

document.getElementById("guSelect").addEventListener("change", e=>draw(e.target.value));
draw("");
</script>
</body></html>"""

options_html = "".join(f'<option value="{g}">{g}</option>' for g in gu_list)
html = (html.replace("__KAKAO__", kakao_key)
            .replace("__OPTIONS__", options_html)
            .replace("__ACC__", _json.dumps(acc_json, ensure_ascii=False))
            .replace("__CW__",  _json.dumps(cw_json, ensure_ascii=False)))

Path("crosswalk_safety_map.html").write_text(html, encoding="utf-8")
print(f"\n저장 완료: crosswalk_safety_map.html")
print(f"브라우저로 열기: open crosswalk_safety_map.html")


지도 표시:
  - 노인 사고: 1,963
  - 신호 없는 횡단보도(사고 200m 내): 12,330 / 전체 신호없음 26,735

저장 완료: crosswalk_safety_map.html
브라우저로 열기: open crosswalk_safety_map.html


## 핵심 발견 (실행 후 채우기)

이 노트북을 실행하면 다음 산출물이 생성됨:

- 노트북 출력 — 자치구별 통계 테이블, 인프라 결핍도 TOP/BOTTOM, 결핍도 vs 사고 랭킹
- **`crosswalk_safety_map.html`** — 사고 점 + 신호 없는 횡단보도 (사고 200m 이내) 지도

**해석 시 유의**
- TAAS 사고는 1년치(2024), 횡단보도 데이터는 2026-03-20 시점 — 시간 단위 다름
- 사고-횡단보도 50m 인접 = "사고가 그 횡단보도에서 났다"가 아니라 **상관 관계** 시그널
- 인구·노인 비율 보정 미반영 — 자치구 절대 비교에 주의

**다음 단계 후보**
- 정류소 종합 위험도(`seoul_elderly_viz.ipynb`)에 인프라 결핍도 4번째 축으로 합산
- 음향신호기 보급률과 노인 인구 밀도(서울 열린데이터광장) 결합
